In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.transform import Rotation, Slerp, RotationSpline

rot_cols = ['rotation_r11','rotation_r12','rotation_r13',
            'rotation_r21','rotation_r22','rotation_r23',
            'rotation_r31','rotation_r32','rotation_r33']

def to_rotation(row):
    R = np.array([[float(row['rotation_r11']), float(row['rotation_r12']), float(row['rotation_r13'])],
                  [float(row['rotation_r21']), float(row['rotation_r22']), float(row['rotation_r23'])],
                  [float(row['rotation_r31']), float(row['rotation_r32']), float(row['rotation_r33'])]])
    U, _, Vt = np.linalg.svd(R)
    return Rotation.from_matrix(U @ Vt)

def windowed_rotation_gt(df, prev_idx, next_idx, new_fns, window=5):
    """
    Independent rotation ground truth: fit a RotationSpline separately on the
    left-side window and right-side window of REAL frames (never the two
    boundary frames alone), extrapolate each to the query times, then blend
    with weight = alpha. This is deliberately NOT the same computation as the
    2-point SLERP used for the 'after' method, so it cannot trivially match it.
    """
    left  = max(0, prev_idx - window + 1)
    right = min(len(df) - 1, next_idx + window - 1)
    left_idx  = list(range(left, prev_idx + 1))
    right_idx = list(range(next_idx, right + 1))

    prev_fn = float(df.iloc[prev_idx]['framenumber'])
    next_fn = float(df.iloc[next_idx]['framenumber'])
    alphas  = (new_fns - prev_fn) / (next_fn - prev_fn)

    left_rows  = df.iloc[left_idx]
    right_rows = df.iloc[right_idx]

    left_times  = left_rows['framenumber'].values.astype(float)
    right_times = right_rows['framenumber'].values.astype(float)
    left_rots   = Rotation.concatenate([to_rotation(r) for _, r in left_rows.iterrows()])
    right_rots  = Rotation.concatenate([to_rotation(r) for _, r in right_rows.iterrows()])

    # Need at least 2 distinct times on each side to fit a spline
    if len(np.unique(left_times)) < 2 or len(np.unique(right_times)) < 2:
        # fall back to plain SLERP if window is too small (rare edge case)
        key_rots = Rotation.concatenate([to_rotation(df.iloc[prev_idx]), to_rotation(df.iloc[next_idx])])
        return Slerp([0.0, 1.0], key_rots)(alphas)

    left_spline  = RotationSpline(left_times, left_rots)
    right_spline = RotationSpline(right_times, right_rots)

    # Extrapolate each spline to the query times (outside its own fitted range)
    left_pred  = left_spline(new_fns)
    right_pred = right_spline(new_fns)

    gt_rots = []
    for k, a in enumerate(alphas):
        pair = Rotation.concatenate([left_pred[k], right_pred[k]])
        gt_rots.append(Slerp([0.0, 1.0], pair)([a])[0])
    return Rotation.concatenate(gt_rots)

def geodesic_deg(Ra, Rb):
    return (Ra.inv() * Rb).magnitude() * 180 / np.pi

def print_comparison_table_fixed(all_scene_results, df, window=5):
    print(f"\n{'='*75}")
    print(f"  {'Scene Change':<20} {'Before (Linear)':<25} {'After (SLERP+Spline)':<25}")
    print(f"  {'':20} {'Trans(m)   Rot(deg)':<25} {'Trans(m)   Rot(deg)':<25}")
    print(f"  {'-'*70}")

    all_tb, all_rb, all_ta, all_ra = [], [], [], []

    for r in all_scene_results:
        prev_fn, next_fn = r['prev_fn'], r['next_fn']
        interp_linear = r['interp_linear']
        interp_slerp  = r['interp_slerp']

        prev_idx = df.index[df['framenumber'] == prev_fn][0]
        next_idx = df.index[df['framenumber'] == next_fn][0]
        new_fns  = interp_linear['framenumber'].values if 'framenumber' in interp_linear \
                   else interp_slerp['framenumber'].values

        gt_rots = windowed_rotation_gt(df, prev_idx, next_idx, new_fns, window=window)

        # Translation GT stays as the linear blend between boundary world positions (unchanged)
        prev_row, next_row = df.iloc[prev_idx], df.iloc[next_idx]

        trans_b, rot_b, trans_a, rot_a = [], [], [], []
        alphas = (new_fns - float(prev_row['framenumber'])) / \
                 (float(next_row['framenumber']) - float(prev_row['framenumber']))

        for k, alpha in enumerate(alphas):
            R_before = to_rotation(interp_linear.iloc[k])   # naive linear blend, re-orthogonalized
            R_after  = to_rotation(interp_slerp.iloc[k])    # 2-point SLERP
            R_gt     = gt_rots[k]                           # independent windowed spline GT

            rot_b.append(geodesic_deg(R_before, R_gt) ** 2)
            rot_a.append(geodesic_deg(R_after,  R_gt) ** 2)

        rmse_rot_before = np.sqrt(np.mean(rot_b))
        rmse_rot_after  = np.sqrt(np.mean(rot_a))

        # keep translation RMSE from earlier logic (already independent — unchanged)
        rmse_trans_before = r.get('trans_rmse_before', np.nan)
        rmse_trans_after  = r.get('trans_rmse_after', np.nan)

        print(f"  fn {prev_fn} -> {next_fn:<12} "
              f"{rmse_trans_before:>7.4f} m {rmse_rot_before:>8.4f} deg   "
              f"{rmse_trans_after:>7.4f} m {rmse_rot_after:>8.4f} deg")

        all_rb.append(rmse_rot_before); all_ra.append(rmse_rot_after)

    avg_rb, avg_ra = np.mean(all_rb), np.mean(all_ra)
    rot_imp = ((avg_rb - avg_ra) / avg_rb * 100) if avg_rb > 0 else 0
    print(f"  {'-'*70}")
    print(f"  Average rotation RMSE — Before: {avg_rb:.4f} deg | After: {avg_ra:.4f} deg")
    print(f"  Rotation improvement: {rot_imp:+.2f}%")

In [ ]:
import pandas as pd
import cv2
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.spatial.transform import Rotation, Slerp, RotationSpline

# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('pumpkin1.csv')

loc_cols     = ['focal_fx','skew_s','principal_cx','principal_cy',
                'rotation_r11','rotation_r12','rotation_r13',
                'rotation_r21','rotation_r22','rotation_r23',
                'rotation_r31','rotation_r32','rotation_r33',
                'translation_t1','translation_t2','translation_t3']
rot_cols     = ['rotation_r11','rotation_r12','rotation_r13',
                'rotation_r21','rotation_r22','rotation_r23',
                'rotation_r31','rotation_r32','rotation_r33']
non_rot_cols = ['focal_fx','skew_s','principal_cx','principal_cy',
                'translation_t1','translation_t2','translation_t3']

# ── 2. Read video frames ───────────────────────────────────────────────────────
cap = cv2.VideoCapture('pumpkin.mp4')
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()

# ── 3. Sync CSV and video length ──────────────────────────────────────────────
min_count = min(len(frames), len(df))
frames    = frames[:min_count]
df        = df.iloc[:min_count].reset_index(drop=True)
print(f"Using (minimum) : {min_count} frames")

# ── 4. Scene change detection (dynamic threshold) ─────────────────────────────
def histogram_diff(f1, f2):
    g1 = cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY)
    g2 = cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY)
    h1 = cv2.calcHist([g1],[0],None,[256],[0,256]); cv2.normalize(h1,h1)
    h2 = cv2.calcHist([g2],[0],None,[256],[0,256]); cv2.normalize(h2,h2)
    return cv2.compareHist(h1, h2, cv2.HISTCMP_BHATTACHARYYA)

diffs         = [histogram_diff(frames[i-1], frames[i]) for i in range(1, len(frames))]
threshold     = np.mean(diffs) + 3 * np.std(diffs)
scene_changes = [i for i, d in enumerate(diffs) if d > threshold]
print(f"Dynamic threshold : {threshold:.4f}")
print(f"Scene changes at  : {scene_changes}")

# ── 5. Localization helpers ───────────────────────────────────────────────────
def to_rotation(row):
    R = np.array([[float(row['rotation_r11']), float(row['rotation_r12']), float(row['rotation_r13'])],
                  [float(row['rotation_r21']), float(row['rotation_r22']), float(row['rotation_r23'])],
                  [float(row['rotation_r31']), float(row['rotation_r32']), float(row['rotation_r33'])]])
    U, _, Vt = np.linalg.svd(R)
    return Rotation.from_matrix(U @ Vt)

def get_translation(row):
    return np.array([float(row['translation_t1']),
                     float(row['translation_t2']),
                     float(row['translation_t3'])])

def camera_to_world(row):
    """World position (m) and euler angles (deg) from a localization row."""
    R_obj = to_rotation(row)
    t     = get_translation(row)
    R_mat = R_obj.as_matrix()
    pos   = -R_mat.T @ t
    euler = Rotation.from_matrix(R_mat.T).as_euler('xyz', degrees=True)
    return pos, euler

def geodesic_deg(Ra, Rb):
    return (Ra.inv() * Rb).magnitude() * 180 / np.pi

# ── 6. BEFORE — linear interpolation of raw parameters ────────────────────────
def linear_interp_loc(prev_row, next_row, alphas):
    rows = []
    for alpha in alphas:
        row = {col: (1-alpha)*float(prev_row[col]) + alpha*float(next_row[col])
               for col in loc_cols}
        rows.append(row)
    return pd.DataFrame(rows)

# ── 7. AFTER — Spline (non-rotation) + SLERP (rotation) ──────────────────────
def slerp_spline_interp_loc(df, prev_idx, next_idx, n=3, window=5):
    left      = max(0, prev_idx - window + 1)
    right     = min(len(df) - 1, next_idx + window - 1)
    left_idx  = list(range(left, prev_idx + 1))
    right_idx = list(range(next_idx, right + 1))

    prev_fn = float(df.iloc[prev_idx]['framenumber'])
    next_fn = float(df.iloc[next_idx]['framenumber'])
    new_fns = np.linspace(prev_fn, next_fn, n + 2)[1:-1]
    alphas  = (new_fns - prev_fn) / (next_fn - prev_fn)

    interp_rows = {'framenumber': new_fns}

    for col in non_rot_cols:
        lr  = df.iloc[left_idx]
        rr  = df.iloc[right_idx]
        csl = CubicSpline(lr['framenumber'].values.astype(float), lr[col].values.astype(float))
        csr = CubicSpline(rr['framenumber'].values.astype(float), rr[col].values.astype(float))
        interp_rows[col] = (1 - alphas) * csl(new_fns) + alphas * csr(new_fns)

    key_rots    = Rotation.concatenate([to_rotation(df.iloc[prev_idx]),
                                        to_rotation(df.iloc[next_idx])])
    slerp_fn    = Slerp([0.0, 1.0], key_rots)
    interp_rots = slerp_fn(alphas)
    for k, Rm in enumerate(interp_rots.as_matrix()):
        for ri, rj, lbl in [(0,0,'r11'),(0,1,'r12'),(0,2,'r13'),
                             (1,0,'r21'),(1,1,'r22'),(1,2,'r23'),
                             (2,0,'r31'),(2,1,'r32'),(2,2,'r33')]:
            interp_rows.setdefault(f'rotation_{lbl}', []).append(Rm[ri,rj])

    return pd.DataFrame(interp_rows), alphas

# ── 8. GROUND TRUTH — translation (linear blend of world positions) ──────────
def translation_ground_truth(pos_prev, pos_next, alphas):
    return [(1 - a) * pos_prev + a * pos_next for a in alphas]

# ── 9. GROUND TRUTH — rotation (INDEPENDENT windowed RotationSpline) ──────────
def windowed_rotation_gt(df, prev_idx, next_idx, new_fns, window=5):
    """
    Independent of the 2-point SLERP used in the 'after' method.
    Fits a RotationSpline on real frames to the left and right of the gap,
    extrapolates each to the query times, then blends by alpha.
    """
    left  = max(0, prev_idx - window + 1)
    right = min(len(df) - 1, next_idx + window - 1)
    left_idx  = list(range(left, prev_idx + 1))
    right_idx = list(range(next_idx, right + 1))

    prev_fn = float(df.iloc[prev_idx]['framenumber'])
    next_fn = float(df.iloc[next_idx]['framenumber'])
    alphas  = (new_fns - prev_fn) / (next_fn - prev_fn)

    left_rows  = df.iloc[left_idx]
    right_rows = df.iloc[right_idx]
    left_times  = left_rows['framenumber'].values.astype(float)
    right_times = right_rows['framenumber'].values.astype(float)

    if len(np.unique(left_times)) < 2 or len(np.unique(right_times)) < 2:
        # fallback for scenes too close to video start/end
        key_rots = Rotation.concatenate([to_rotation(df.iloc[prev_idx]), to_rotation(df.iloc[next_idx])])
        return Slerp([0.0, 1.0], key_rots)(alphas)

    left_rots  = Rotation.concatenate([to_rotation(r) for _, r in left_rows.iterrows()])
    right_rots = Rotation.concatenate([to_rotation(r) for _, r in right_rows.iterrows()])

    left_spline  = RotationSpline(left_times, left_rots)
    right_spline = RotationSpline(right_times, right_rots)

    left_pred  = left_spline(new_fns)
    right_pred = right_spline(new_fns)

    gt_rots = []
    for k, a in enumerate(alphas):
        pair = Rotation.concatenate([left_pred[k], right_pred[k]])
        gt_rots.append(Slerp([0.0, 1.0], pair)([a])[0])
    return Rotation.concatenate(gt_rots)

# ── 10. Main loop: compute before/after error vs independent ground truth ─────
all_scene_results = []

for sc in scene_changes:
    prev_idx = sc
    next_idx = sc + 1
    if next_idx >= len(frames) or next_idx >= len(df):
        continue

    prev_row = df.iloc[prev_idx]
    next_row = df.iloc[next_idx]
    prev_fn  = df.iloc[prev_idx]['framenumber']
    next_fn  = df.iloc[next_idx]['framenumber']

    # Before: linear param interpolation
    interp_slerp, alphas = slerp_spline_interp_loc(df, prev_idx, next_idx, n=3, window=5)
    interp_linear         = linear_interp_loc(prev_row, next_row, alphas)
    new_fns               = interp_slerp['framenumber'].values

    # Ground truth: translation (linear world blend) + rotation (windowed spline)
    pos_prev, _ = camera_to_world(prev_row)
    pos_next, _ = camera_to_world(next_row)
    gt_trans    = translation_ground_truth(pos_prev, pos_next, alphas)
    gt_rots     = windowed_rotation_gt(df, prev_idx, next_idx, new_fns, window=5)

    trans_err_before, trans_err_after = [], []
    rot_err_before,   rot_err_after   = [], []

    for k in range(len(interp_linear)):
        pos_b, _ = camera_to_world(interp_linear.iloc[k])
        pos_a, _ = camera_to_world(interp_slerp.iloc[k])
        R_b = to_rotation(interp_linear.iloc[k])
        R_a = to_rotation(interp_slerp.iloc[k])

        trans_err_before.append(np.linalg.norm(pos_b - gt_trans[k]) ** 2)
        trans_err_after.append(np.linalg.norm(pos_a - gt_trans[k]) ** 2)
        rot_err_before.append(geodesic_deg(R_b, gt_rots[k]) ** 2)
        rot_err_after.append(geodesic_deg(R_a, gt_rots[k]) ** 2)

    rmse_trans_before = np.sqrt(np.mean(trans_err_before))
    rmse_trans_after   = np.sqrt(np.mean(trans_err_after))
    rmse_rot_before    = np.sqrt(np.mean(rot_err_before))
    rmse_rot_after     = np.sqrt(np.mean(rot_err_after))

    all_scene_results.append({
        'prev_fn': prev_fn, 'next_fn': next_fn,
        'trans_rmse_before': rmse_trans_before, 'trans_rmse_after': rmse_trans_after,
        'rot_rmse_before'  : rmse_rot_before,   'rot_rmse_after'  : rmse_rot_after,
    })

# ── 11. Print table in the same format as before ──────────────────────────────
print(f"\n{'='*75}")
print(f"  {'Scene Change':<20} {'Before Interpolation':<25} {'After Interpolation':<25}")
print(f"  {'(fn A -> fn B)':<20} {'Linear':<25} {'Spline + SLERP':<25}")
print(f"  {'-'*70}")
print(f"  {'':20} {'Trans (m)   Rot (deg)':<25} {'Trans (m)   Rot (deg)':<25}")
print(f"  {'-'*70}")

for r in all_scene_results:
    scene_label  = f"fn {r['prev_fn']} -> {r['next_fn']}"
    before_label = f"{r['trans_rmse_before']:.4f} m  {r['rot_rmse_before']:.4f} deg"
    after_label  = f"{r['trans_rmse_after']:.4f} m  {r['rot_rmse_after']:.4f} deg"
    print(f"  {scene_label:<20} {before_label:<25} {after_label:<25}")

print(f"  {'-'*70}")

avg_tb = np.mean([r['trans_rmse_before'] for r in all_scene_results])
avg_ta = np.mean([r['trans_rmse_after']  for r in all_scene_results])
avg_rb = np.mean([r['rot_rmse_before']   for r in all_scene_results])
avg_ra = np.mean([r['rot_rmse_after']    for r in all_scene_results])

print(f"  {'AVERAGE':<20} "
      f"{avg_tb:.4f} m  {avg_rb:.4f} deg     "
      f"{avg_ta:.4f} m  {avg_ra:.4f} deg")
print(f"  {'-'*70}")

trans_imp = ((avg_tb - avg_ta) / avg_tb * 100) if avg_tb > 0 else 0
rot_imp   = ((avg_rb - avg_ra) / avg_rb * 100) if avg_rb > 0 else 0

print(f"\n  Improvement after interpolation:")
print(f"    Translation : {trans_imp:+.2f}%  ({'reduced' if trans_imp > 0 else 'increased'})")
print(f"    Rotation    : {rot_imp:+.2f}%  ({'reduced' if rot_imp > 0 else 'increased'})")
print(f"{'='*75}")

In [ ]:
import pandas as pd
import cv2
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.spatial.transform import Rotation, Slerp, RotationSpline

# ── 1. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('pumpkin1.csv')

loc_cols     = ['focal_fx','skew_s','principal_cx','principal_cy',
                'rotation_r11','rotation_r12','rotation_r13',
                'rotation_r21','rotation_r22','rotation_r23',
                'rotation_r31','rotation_r32','rotation_r33',
                'translation_t1','translation_t2','translation_t3']
rot_cols     = ['rotation_r11','rotation_r12','rotation_r13',
                'rotation_r21','rotation_r22','rotation_r23',
                'rotation_r31','rotation_r32','rotation_r33']
non_rot_cols = ['focal_fx','skew_s','principal_cx','principal_cy',
                'translation_t1','translation_t2','translation_t3']

# ── 2. Read video frames ───────────────────────────────────────────────────────
cap = cv2.VideoCapture('pumpkin.mp4')
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()

# ── 3. Sync CSV and video length ──────────────────────────────────────────────
min_count = min(len(frames), len(df))
frames    = frames[:min_count]
df        = df.iloc[:min_count].reset_index(drop=True)
print(f"Using (minimum) : {min_count} frames")

# ── 4. Scene change detection (dynamic threshold) ─────────────────────────────
def histogram_diff(f1, f2):
    g1 = cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY)
    g2 = cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY)
    h1 = cv2.calcHist([g1],[0],None,[256],[0,256]); cv2.normalize(h1,h1)
    h2 = cv2.calcHist([g2],[0],None,[256],[0,256]); cv2.normalize(h2,h2)
    return cv2.compareHist(h1, h2, cv2.HISTCMP_BHATTACHARYYA)

diffs         = [histogram_diff(frames[i-1], frames[i]) for i in range(1, len(frames))]
threshold     = np.mean(diffs) + 3 * np.std(diffs)
scene_changes = [i for i, d in enumerate(diffs) if d > threshold]
print(f"Dynamic threshold : {threshold:.4f}")
print(f"Scene changes at  : {scene_changes}")

# ── 5. Localization helpers ───────────────────────────────────────────────────
def to_rotation(row):
    R = np.array([[float(row['rotation_r11']), float(row['rotation_r12']), float(row['rotation_r13'])],
                  [float(row['rotation_r21']), float(row['rotation_r22']), float(row['rotation_r23'])],
                  [float(row['rotation_r31']), float(row['rotation_r32']), float(row['rotation_r33'])]])
    U, _, Vt = np.linalg.svd(R)
    return Rotation.from_matrix(U @ Vt)

def get_translation(row):
    return np.array([float(row['translation_t1']),
                     float(row['translation_t2']),
                     float(row['translation_t3'])])

def camera_to_world(row):
    R_obj = to_rotation(row)
    t     = get_translation(row)
    R_mat = R_obj.as_matrix()
    pos   = -R_mat.T @ t
    euler = Rotation.from_matrix(R_mat.T).as_euler('xyz', degrees=True)
    return pos, euler

def geodesic_deg(Ra, Rb):
    return (Ra.inv() * Rb).magnitude() * 180 / np.pi

# ── 6. BEFORE — linear interpolation of raw parameters ────────────────────────
def linear_interp_loc(prev_row, next_row, alphas):
    rows = []
    for alpha in alphas:
        row = {col: (1-alpha)*float(prev_row[col]) + alpha*float(next_row[col])
               for col in loc_cols}
        rows.append(row)
    return pd.DataFrame(rows)

# ── 7. AFTER — Spline (non-rotation) + SLERP (rotation) ──────────────────────
def slerp_spline_interp_loc(df, prev_idx, next_idx, n, window):
    left      = max(0, prev_idx - window + 1)
    right     = min(len(df) - 1, next_idx + window - 1)
    left_idx  = list(range(left, prev_idx + 1))
    right_idx = list(range(next_idx, right + 1))

    prev_fn = float(df.iloc[prev_idx]['framenumber'])
    next_fn = float(df.iloc[next_idx]['framenumber'])
    new_fns = np.linspace(prev_fn, next_fn, n + 2)[1:-1]
    alphas  = (new_fns - prev_fn) / (next_fn - prev_fn)

    interp_rows = {'framenumber': new_fns}

    for col in non_rot_cols:
        lr  = df.iloc[left_idx]
        rr  = df.iloc[right_idx]
        csl = CubicSpline(lr['framenumber'].values.astype(float), lr[col].values.astype(float))
        csr = CubicSpline(rr['framenumber'].values.astype(float), rr[col].values.astype(float))
        interp_rows[col] = (1 - alphas) * csl(new_fns) + alphas * csr(new_fns)

    key_rots    = Rotation.concatenate([to_rotation(df.iloc[prev_idx]),
                                        to_rotation(df.iloc[next_idx])])
    slerp_fn    = Slerp([0.0, 1.0], key_rots)
    interp_rots = slerp_fn(alphas)
    for k, Rm in enumerate(interp_rots.as_matrix()):
        for ri, rj, lbl in [(0,0,'r11'),(0,1,'r12'),(0,2,'r13'),
                             (1,0,'r21'),(1,1,'r22'),(1,2,'r23'),
                             (2,0,'r31'),(2,1,'r32'),(2,2,'r33')]:
            interp_rows.setdefault(f'rotation_{lbl}', []).append(Rm[ri,rj])

    return pd.DataFrame(interp_rows), alphas

# ── 8. GROUND TRUTH — translation (linear blend of world positions) ──────────
def translation_ground_truth(pos_prev, pos_next, alphas):
    return [(1 - a) * pos_prev + a * pos_next for a in alphas]

# ── 9. GROUND TRUTH — rotation (INDEPENDENT windowed RotationSpline) ──────────
def windowed_rotation_gt(df, prev_idx, next_idx, new_fns, window):
    left  = max(0, prev_idx - window + 1)
    right = min(len(df) - 1, next_idx + window - 1)
    left_idx  = list(range(left, prev_idx + 1))
    right_idx = list(range(next_idx, right + 1))

    prev_fn = float(df.iloc[prev_idx]['framenumber'])
    next_fn = float(df.iloc[next_idx]['framenumber'])
    alphas  = (new_fns - prev_fn) / (next_fn - prev_fn)

    left_rows  = df.iloc[left_idx]
    right_rows = df.iloc[right_idx]
    left_times  = left_rows['framenumber'].values.astype(float)
    right_times = right_rows['framenumber'].values.astype(float)

    if len(np.unique(left_times)) < 2 or len(np.unique(right_times)) < 2:
        key_rots = Rotation.concatenate([to_rotation(df.iloc[prev_idx]), to_rotation(df.iloc[next_idx])])
        return Slerp([0.0, 1.0], key_rots)(alphas)

    left_rots  = Rotation.concatenate([to_rotation(r) for _, r in left_rows.iterrows()])
    right_rots = Rotation.concatenate([to_rotation(r) for _, r in right_rows.iterrows()])

    left_spline  = RotationSpline(left_times, left_rots)
    right_spline = RotationSpline(right_times, right_rots)

    left_pred  = left_spline(new_fns)
    right_pred = right_spline(new_fns)

    gt_rots = []
    for k, a in enumerate(alphas):
        pair = Rotation.concatenate([left_pred[k], right_pred[k]])
        gt_rots.append(Slerp([0.0, 1.0], pair)([a])[0])
    return Rotation.concatenate(gt_rots)

# ── 10. Compute RMSE (before/after) for ONE window size ───────────────────────
def compute_scene_results(df, scene_changes, frames, window, n=3):
    results = []
    for sc in scene_changes:
        prev_idx = sc
        next_idx = sc + 1
        if next_idx >= len(frames) or next_idx >= len(df):
            continue

        prev_row = df.iloc[prev_idx]
        next_row = df.iloc[next_idx]
        prev_fn  = df.iloc[prev_idx]['framenumber']
        next_fn  = df.iloc[next_idx]['framenumber']

        interp_slerp, alphas = slerp_spline_interp_loc(df, prev_idx, next_idx, n=n, window=window)
        interp_linear         = linear_interp_loc(prev_row, next_row, alphas)
        new_fns                = interp_slerp['framenumber'].values

        pos_prev, _ = camera_to_world(prev_row)
        pos_next, _ = camera_to_world(next_row)
        gt_trans    = translation_ground_truth(pos_prev, pos_next, alphas)
        gt_rots     = windowed_rotation_gt(df, prev_idx, next_idx, new_fns, window=window)

        trans_err_before, trans_err_after = [], []
        rot_err_before,   rot_err_after   = [], []

        for k in range(len(interp_linear)):
            pos_b, _ = camera_to_world(interp_linear.iloc[k])
            pos_a, _ = camera_to_world(interp_slerp.iloc[k])
            R_b = to_rotation(interp_linear.iloc[k])
            R_a = to_rotation(interp_slerp.iloc[k])

            trans_err_before.append(np.linalg.norm(pos_b - gt_trans[k]) ** 2)
            trans_err_after.append(np.linalg.norm(pos_a - gt_trans[k]) ** 2)
            rot_err_before.append(geodesic_deg(R_b, gt_rots[k]) ** 2)
            rot_err_after.append(geodesic_deg(R_a, gt_rots[k]) ** 2)

        results.append({
            'prev_fn': prev_fn, 'next_fn': next_fn,
            'trans_rmse_before': np.sqrt(np.mean(trans_err_before)),
            'trans_rmse_after' : np.sqrt(np.mean(trans_err_after)),
            'rot_rmse_before'  : np.sqrt(np.mean(rot_err_before)),
            'rot_rmse_after'   : np.sqrt(np.mean(rot_err_after)),
        })
    return results

# ── 11. Print table for one window size ────────────────────────────────────────
def print_window_table(results, window):
    print(f"\n{'='*75}")
    print(f"  WINDOW SIZE = {window}")
    print(f"{'='*75}")
    print(f"  {'Scene Change':<20} {'Before Interpolation':<25} {'After Interpolation':<25}")
    print(f"  {'(fn A -> fn B)':<20} {'Linear':<25} {'Spline + SLERP':<25}")
    print(f"  {'-'*70}")
    print(f"  {'':20} {'Trans (m)   Rot (deg)':<25} {'Trans (m)   Rot (deg)':<25}")
    print(f"  {'-'*70}")

    for r in results:
        scene_label  = f"fn {r['prev_fn']} -> {r['next_fn']}"
        before_label = f"{r['trans_rmse_before']:.4f} m  {r['rot_rmse_before']:.4f} deg"
        after_label  = f"{r['trans_rmse_after']:.4f} m  {r['rot_rmse_after']:.4f} deg"
        print(f"  {scene_label:<20} {before_label:<25} {after_label:<25}")

    print(f"  {'-'*70}")

    avg_tb = np.mean([r['trans_rmse_before'] for r in results])
    avg_ta = np.mean([r['trans_rmse_after']  for r in results])
    avg_rb = np.mean([r['rot_rmse_before']   for r in results])
    avg_ra = np.mean([r['rot_rmse_after']    for r in results])

    print(f"  {'AVERAGE':<20} "
          f"{avg_tb:.4f} m  {avg_rb:.4f} deg     "
          f"{avg_ta:.4f} m  {avg_ra:.4f} deg")
    print(f"  {'-'*70}")

    trans_imp = ((avg_tb - avg_ta) / avg_tb * 100) if avg_tb > 0 else 0
    rot_imp   = ((avg_rb - avg_ra) / avg_rb * 100) if avg_rb > 0 else 0

    print(f"\n  Improvement after interpolation (window={window}):")
    print(f"    Translation : {trans_imp:+.2f}%  ({'reduced' if trans_imp > 0 else 'increased'})")
    print(f"    Rotation    : {rot_imp:+.2f}%  ({'reduced' if rot_imp > 0 else 'increased'})")

    return {
        'window': window,
        'avg_trans_before': avg_tb, 'avg_trans_after': avg_ta, 'trans_improvement_pct': trans_imp,
        'avg_rot_before'  : avg_rb, 'avg_rot_after'  : avg_ra, 'rot_improvement_pct'  : rot_imp,
    }

# ── 12. Sweep window sizes 3 to 8 ───────────────────────────────────────────────
window_summary = []

for window in range(3, 9):   # 3,4,5,6,7,8
    results = compute_scene_results(df, scene_changes, frames, window=window, n=3)
    if len(results) == 0:
        print(f"\nWindow={window}: no valid scene changes to evaluate, skipping.")
        continue
    summary = print_window_table(results, window)
    window_summary.append(summary)

# ── 13. Summary across all window sizes ─────────────────────────────────────────
print(f"\n{'='*75}")
print(f"  SUMMARY ACROSS WINDOW SIZES (3 - 8)")
print(f"{'='*75}")
print(f"  {'Window':<8} {'Trans Before(m)':>16} {'Trans After(m)':>16} {'Trans Δ%':>10}   "
      f"{'Rot Before(deg)':>16} {'Rot After(deg)':>16} {'Rot Δ%':>10}")
print(f"  {'-'*95}")
for s in window_summary:
    print(f"  {s['window']:<8} "
          f"{s['avg_trans_before']:>16.4f} {s['avg_trans_after']:>16.4f} {s['trans_improvement_pct']:>+9.2f}%   "
          f"{s['avg_rot_before']:>16.4f} {s['avg_rot_after']:>16.4f} {s['rot_improvement_pct']:>+9.2f}%")
print(f"  {'-'*95}")

best_trans_window = max(window_summary, key=lambda s: s['trans_improvement_pct'])
best_rot_window    = max(window_summary, key=lambda s: s['rot_improvement_pct'])
print(f"\n  Best window for translation improvement : {best_trans_window['window']} "
      f"({best_trans_window['trans_improvement_pct']:+.2f}%)")
print(f"  Best window for rotation improvement    : {best_rot_window['window']} "
      f"({best_rot_window['rot_improvement_pct']:+.2f}%)")
print(f"{'='*75}")